# SPA for Maize MIR Regression



## Import required libraries

In [ ]:
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import KFold
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore")

## Data loading and standardization

In [ ]:
X_raw = np.loadtxt("dataF_final.csv", delimiter=",")
Y_raw = np.loadtxt("dataC_final.csv", delimiter=",")
if Y_raw.ndim == 1:
    Y_raw = Y_raw.reshape(-1, 1)

scaler_x = StandardScaler()
scaler_y = StandardScaler()
X = scaler_x.fit_transform(X_raw)
Y = scaler_y.fit_transform(Y_raw)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

## Cross-validated RMSE and model evaluation helpers

In [ ]:
def cv_rmse(X_subset, y, n_components=20):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmses = []
    comp = min(n_components, X_subset.shape[1], X_subset.shape[0] - 1)
    model = PLSRegression(n_components=comp)
    for tr, te in kf.split(X_subset):
        model.fit(X_subset[tr], y[tr])
        pred = model.predict(X_subset[te]).ravel()
        rmses.append(np.sqrt(mean_squared_error(y[te], pred)))
    return np.mean(rmses)


def evaluate_models(X_selected, y_target, target_number, method_name):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    test_models = {
        "PLSR": PLSRegression(n_components=min(20, X_selected.shape[1], X_selected.shape[0] - 1)),
        "Ridge": Ridge(alpha=1),
        "Lasso": Lasso(alpha=0.001, max_iter=10000),
        "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.001, max_iter=10000)
    }

    rows = []
    predictions = {}

    for name, model_template in test_models.items():
        fold_r2 = []
        fold_rmse = []
        y_true_all = []
        y_pred_all = []

        for train_idx, test_idx in kf.split(X_selected):
            X_train, X_test = X_selected[train_idx], X_selected[test_idx]
            y_train, y_test = y_target[train_idx], y_target[test_idx]

            model = clone(model_template)
            if isinstance(model, PLSRegression):
                model.n_components = min(model.n_components, X_train.shape[1], len(train_idx) - 1)

            model.fit(X_train, y_train)
            pred = model.predict(X_test).ravel()

            y_true_all.extend(y_test)
            y_pred_all.extend(pred)

            fold_r2.append(r2_score(y_test, pred))
            fold_rmse.append(np.sqrt(mean_squared_error(y_test, pred)))

        rows.append({
            "Method": method_name,
            "Target": target_number,
            "Model": name,
            "R2_test": np.mean(fold_r2),
            "RMSE_test": np.mean(fold_rmse),
            "R2_all_folds": r2_score(y_true_all, y_pred_all),
            "RMSE_all_folds": np.sqrt(mean_squared_error(y_true_all, y_pred_all)),
            "Features": X_selected.shape[1]
        })
        predictions[(target_number, name)] = (np.array(y_true_all), np.array(y_pred_all))

    return rows, predictions

## Feature selection method

In [ ]:
def _spa_sequence(X_pool, start_position, max_features):
    selected_positions = [start_position]
    remaining = list(range(X_pool.shape[1]))
    remaining.remove(start_position)

    for _ in range(1, max_features):
        if not remaining:
            break

        S = X_pool[:, selected_positions]
        Q, _ = np.linalg.qr(S, mode="reduced")
        X_remaining = X_pool[:, remaining]
        residual = X_remaining - Q @ (Q.T @ X_remaining)
        norms = np.sum(residual ** 2, axis=0)

        next_position = remaining[int(np.argmax(norms))]
        selected_positions.append(next_position)
        remaining.remove(next_position)

    return selected_positions


def spa(X, y_single, random_state=42):
   
    n_samples, n_features = X.shape
    y_centered = y_single - y_single.mean()

    corr = np.abs(X.T @ y_centered) / ((n_samples - 1) * (X.std(axis=0) + 1e-12) * (y_centered.std() + 1e-12))
    pool_size = min(500, n_features)
    pool_idx = np.argsort(-corr)[:pool_size]
    X_pool = X[:, pool_idx]

  
    start_positions = list(range(min(12, pool_size)))
    max_features = min(120, pool_size, n_samples - 2)

    candidate_sizes = [5, 10, 15, 20, 30, 40, 50, 60, 80, 100, 120]
    candidate_sizes = [k for k in candidate_sizes if k <= max_features]

    best_idx = pool_idx[:candidate_sizes[0]]
    best_rmse = float("inf")
    best_start = None
    best_size = None

    for start_pos in start_positions:
        sequence_positions = _spa_sequence(X_pool, start_pos, max_features=max_features)
        sequence_idx = pool_idx[sequence_positions]

        for k in candidate_sizes:
            current_idx = sequence_idx[:k]
            current_rmse = cv_rmse(X[:, current_idx], y_single, n_components=20)
            if current_rmse < best_rmse:
                best_rmse = current_rmse
                best_idx = current_idx.copy()
                best_start = int(pool_idx[start_pos])
                best_size = k

    info = {"best_start_wavelength_index": best_start, "best_size": best_size, "cv_rmse": best_rmse, "pool_size": pool_size}
    return np.sort(best_idx), corr, info

## Run feature selection and model evaluation

In [ ]:
results_list_spa = []
all_predictions_spa = {}
selected_features_spa = {}
spa_scores = {}
spa_infos = {}

print(f"{'Target':<8} | {'Model':<12} | {'R2_test':<10} | {'RMSE_test':<10} | {'R2_all':<10} | {'Features':<8}")
print("-" * 90)

for t in range(Y.shape[1]):
    print(f"Optimizing Target {t+1} with SPA...")
    start = time.time()

    y_target = Y[:, t]
    best_idx, scores, info = spa(X, y_target, random_state=42 + t)
    X_selected = X[:, best_idx]

    selected_features_spa[t + 1] = best_idx
    spa_scores[t + 1] = scores
    spa_infos[t + 1] = info

    rows, preds = evaluate_models(X_selected, y_target, t + 1, "SPA")
    results_list_spa.extend(rows)
    all_predictions_spa.update(preds)

    for row in rows:
        print(f"T{row['Target']:<6} | {row['Model']:<12} | {row['R2_test']:.4f}   | {row['RMSE_test']:.4f}   | {row['R2_all_folds']:.4f}   | {row['Features']:<8}")

    print(f"SPA info: {info}")
    print(f"RunTime: {time.time() - start:.2f}s")

results_df_spa = pd.DataFrame(results_list_spa)
results_df_spa.to_csv("SPA_results.csv", index=False)
results_df_spa

## Best model per target

In [ ]:
best_models_spa = (
    results_df_spa.sort_values("RMSE_test")
    .groupby("Target")
    .first()
    .reset_index()
)
best_models_spa.to_csv("SPA_best_models.csv", index=False)
best_models_spa

## Plot predicted vs observed for best models

In [ ]:
def plot_best_models(best_models, all_predictions):
    cols = 3
    rows = int(np.ceil(len(best_models) / cols))
    plt.figure(figsize=(14, 4 * rows))

    for i, row in best_models.iterrows():
        t = row["Target"]
        model = row["Model"]
        y_true, y_pred = all_predictions[(t, model)]

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)

        plt.subplot(rows, cols, i + 1)
        plt.scatter(y_true, y_pred, s=12)

        min_v = min(y_true.min(), y_pred.min())
        max_v = max(y_true.max(), y_pred.max())
        plt.plot([min_v, max_v], [min_v, max_v], "r--")

        plt.title(f"T{t} - {model} RMSE={rmse:.3f}, R²={r2:.3f}")
        plt.xlabel("Observed")
        plt.ylabel("Predicted")
        plt.grid()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_best_models(best_models_spa, all_predictions_spa)